
# Nutrition50 LLM Results vs Ground Truth

Compare Nutrition50 macro predictions from the Food-Qwen model with ground truth values.



## Setup
Load dependencies, configure display, and point to CSV assets in `llms_approach/`.


In [7]:

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option('display.max_rows', 10)
pd.set_option('display.precision', 3)

BASE_DIR = Path('.').resolve()
LLM_ROOT = BASE_DIR

PREDICTIONS_CSV = LLM_ROOT / 'nutrition50_llm_macros.csv'
VALIDATION_RAW = LLM_ROOT / 'nutrition50_validation.csv'
GROUND_TRUTH_CSV = LLM_ROOT / 'groundtruth.csv'

if not PREDICTIONS_CSV.exists():
    raise FileNotFoundError(f"Predictions CSV missing: {PREDICTIONS_CSV}")
if not VALIDATION_RAW.exists():
    raise FileNotFoundError(f"Validation CSV missing: {VALIDATION_RAW}")

print('Predictions CSV:', PREDICTIONS_CSV)
print('Original validation CSV:', VALIDATION_RAW)
print('Ground truth will be saved to:', GROUND_TRUTH_CSV)


Predictions CSV: /home/chahar/food_new/llms_approach/nutrition50_llm_macros.csv
Original validation CSV: /home/chahar/food_new/llms_approach/nutrition50_validation.csv
Ground truth will be saved to: /home/chahar/food_new/llms_approach/groundtruth.csv



## Prepare Ground Truth
The raw `nutrition50_validation.csv` lacks headers and contains additional ingredient columns. We retain the first six columns (dish_id, calories, mass, carbs, protein, fat) and assign headers.


In [8]:

raw_df = pd.read_csv(VALIDATION_RAW, header=None)
print('Raw validation shape:', raw_df.shape)

required_cols = ['dish_id', 'calories', 'mass', 'carbs', 'protein', 'fat']
if raw_df.shape[1] < len(required_cols):
    raise ValueError('Validation CSV does not contain enough columns to build ground truth.')

ground_truth_df = raw_df.iloc[:, :len(required_cols)].copy()
ground_truth_df.columns = required_cols

ground_truth_df.to_csv(GROUND_TRUTH_CSV, index=False)
print('Saved ground truth CSV to', GROUND_TRUTH_CSV)

ground_truth_df.head()


Raw validation shape: (50, 41)
Saved ground truth CSV to /home/chahar/food_new/llms_approach/groundtruth.csv


,dish_id,calories,mass,carbs,protein,fat
0,dish_1556572657,41.40,36.0,3.852,2.268,0.288
1,dish_1556573514,6.44,23.0,0.092,1.219,0.506
2,dish_1556575014,71.30,62.0,6.634,3.906,0.496
3,dish_1556575083,27.52,64.0,0.192,5.760,2.176
4,dish_1556575124,4.48,28.0,0.056,0.952,0.196



## Load Predictions
Read the LLM macro estimates for comparison.


In [9]:

preds_df = pd.read_csv(PREDICTIONS_CSV)
print('Predictions shape:', preds_df.shape)
preds_df.head()


Predictions shape: (50, 6)


,dish_id,calories,mass,carbs,protein,fat
0,dish_1556572657,30.0,18.9,1.5,1.5,1.5
1,dish_1556573514,19.0,10.5,1.5,1.5,1.5
2,dish_1556575014,38.0,19.5,11.5,1.5,1.5
3,dish_1556575083,49.0,10.5,1.5,1.5,1.5
4,dish_1556575124,10.0,293.4,11.1,1.1,1.1



## Align & Compute Metrics
Join on `dish_id` and compute MAE, RMSE, MAPE (skipping zero ground truth entries) and correlation.


In [10]:

merged_df = ground_truth_df.merge(preds_df, on='dish_id', how='inner', suffixes=('_gt', '_pred'))
print('Merged rows:', len(merged_df))

metrics = []
for metric in ['calories', 'mass', 'carbs', 'protein', 'fat']:
    gt = merged_df[f'{metric}_gt'].astype(float)
    pred = merged_df[f'{metric}_pred'].astype(float)

    diff = pred - gt
    mae = diff.abs().mean()
    rmse = np.sqrt((diff**2).mean())

    valid = gt.replace({0: np.nan})
    mape = (diff.abs() / valid).dropna().mean() * 100 if not valid.isna().all() else np.nan
    corr = np.corrcoef(gt, pred)[0, 1] if gt.std() > 0 and pred.std() > 0 else np.nan

    metrics.append({
        'metric': metric,
        'MAE': mae,
        'RMSE': rmse,
        'MAPE_percent': mape,
        'Correlation': corr,
    })

metrics_df = pd.DataFrame(metrics)
metrics_df


Merged rows: 50


,metric,MAE,RMSE,MAPE_percent,Correlation
0,calories,78.149,118.741,187.837,0.645
1,mass,324.864,666.939,383.948,0.547
2,carbs,14.915,29.929,5261.449,0.363
3,protein,5.980,8.213,496.399,0.573
4,fat,6.553,9.559,607.208,0.429



## Preview Errors
Inspect per-dish residuals to diagnose where predictions diverge.


In [6]:

for metric in ['calories', 'mass', 'carbs', 'protein', 'fat']:
    merged_df[f'{metric}_error'] = merged_df[f'{metric}_pred'] - merged_df[f'{metric}_gt']

error_cols = ['dish_id'] + [f'{metric}_error' for metric in ['calories', 'mass', 'carbs', 'protein', 'fat']]
merged_df[error_cols].head()


,dish_id,calories_error,mass_error,carbs_error,protein_error,fat_error
0,dish_1556572657,-11.40,-17.1,-2.352,-0.768,1.212
1,dish_1556573514,12.56,-12.5,1.408,0.281,0.994
2,dish_1556575014,-33.30,-42.5,4.866,-2.406,1.004
3,dish_1556575083,21.48,-53.5,1.308,-4.260,-0.676
4,dish_1556575124,5.52,265.4,11.044,0.148,0.904
